# Grid Up Datathon — 01 · Keşif

**Takım:** _(takım adını yaz)_ · **Tarih:** _(gün)_

## Problem neden zor?

Elektrik kesintisi tahmininin üç yapısal zorluğunu tahmin etmedik, **ölçtük**:
yarışmadan önce hattın tamamını 68.257 gerçek GDZ kesinti kaydında prova ettik
(İzmir + Manisa, 47 ilçe, 2021-05 → 2022-08, saat damgalı olay kaydı). Bu
notebook'taki her sayı o provanın çıktısıdır; yanında hangi betikle ölçüldüğü yazar.

1. **Sıfır-şişkin hedef.** Gerçek ilçe × gün panelinde günlerin %35.0'ında hiç
   kesinti yok (`scripts/benchmark_gercek.py`). Ortalamayı optimize eden model hem
   sıfırları hem kesintileri ıskalar; metrik, kayıp ve model buna göre seçilmeli.
2. **Gürültü.** Aynı modelin fold skorları 150.8 → 461.4 dk arasında salınıyor
   (`scripts/real_data_rehearsal.py`). Tek fold'a — veya tek skora — bakan her
   karar yanıltıcıdır.
3. **Mekânsal yapı.** Fırtına ilçe sınırı tanımaz; komşu ilçeler aynı gün arızalanır.
   Coğrafya sinyal kaynağıdır ama aynı zamanda sızıntı riskidir: komşu ilçenin
   geçmişi de ancak tahmin ufku kadar kaydırılarak kullanılabilir.

Bu notebook üç karar üretir: **hedef + panel tanımı**, **doğrulama şeması** ve
**sızıntı duvarı**. Sonraki 12 günün her deneyi bu üç karara yaslanır.

In [ ]:
import sys
from pathlib import Path

# Kaggle'da: /kaggle/input/<yarisma>/  · yerelde: data/raw/
IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    # DIKKAT: gridup Kaggle imajinda KURULU DEGILDIR. Onceki surumde sys.path
    # yalnizca YERELDE ayarlaniyordu; Kaggle'da 'import gridup' ModuleNotFound
    # veriyordu. Once offline paket dataset'indeki wheel'i kur, o yoksa ham
    # kaynagi sys.path'e ekle. (Path.glob, glob.glob degil: juri notebook'u
    # ruff'tan geciyor -- PTH207.)
    import subprocess

    _whl = sorted(Path("/kaggle/input").glob("*/gridup-*.whl"))
    if _whl:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
             str(_whl[0]), "-q"],
            check=False,
        )
    else:
        for _src in Path("/kaggle/input").glob("*/src"):
            sys.path.insert(0, str(_src))
else:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gridup import (
    build_panel, environment_report, panel_coverage, profile, read_any, set_global_seed,
)
from gridup.compat import categorical_columns
from gridup.profiling import quick_look
from gridup.turkish import codepoints, has_combining_dot, join_key, strip_qualifier
from gridup.validation import (
    forecast_geometry, leakage_report, purged_time_series_split, suggest_scheme,
)

set_global_seed(42)

# Ortami yazdir -- juri tekrarlanabilirlige bakiyor, bu ucuz bir puan.
for key, value in environment_report().items():
    print(f"{key:<26} {value}")

## 1 · Veriyi oku — neden `read_any`

Türk kurum dosyaları `cp1254` kodlama, `;` ayırıcı ve ondalık `,` ile gelebilir;
düz `pd.read_csv` bunları sessizce bozar. `read_any` kodlamayı ve ayırıcıyı
**kanıtlayarak** seçer. Gerçek GDZ dosyasında ölçülen: kodlama `utf-8-sig` çıktı
ve `İL → il` dahil 8 kolon adı normalize edildi (`scripts/real_data_rehearsal.py`).
Yani bu bir varsayım değil, ilk gerçek dosyada karşılaştığımız davranış.

In [ ]:
DATA_DIR = Path("/kaggle/input/GRID-UP-YARISMA-SLUG") if IS_KAGGLE else Path("../data/raw")

train = read_any(DATA_DIR / "train.csv")
test  = read_any(DATA_DIR / "test.csv")

try:
    sample_submission = read_any(DATA_DIR / "sample_submission.csv")
    print("sample_submission kolonları:", list(sample_submission.columns))
except FileNotFoundError:
    sample_submission = None
    print("sample_submission bulunamadı — dosya adlarını kontrol et:")
    print(sorted(p.name for p in DATA_DIR.glob("*")))

print(f"\ntrain {train.shape}   test {test.shape}")
train.head()

## 2 · Otomatik profil

Tek çağrı, elle iki saat sürecek keşfin yerine geçer: çarpıklık, sıfır yığılması,
ID-benzeri kolonlar, train/test şema farkı (= sızıntı adayı) ve birleşik nokta
(U+0307) taşıyan Türkçe kolonlar işaretlenir. Amaç grafik biriktirmek değil,
**karar listesi** çıkarmaktır.

In [ ]:
# TODO: hedef kolon adini veri geldiginde doldur
TARGET = "HEDEF_KOLON"

dataset_profile = profile(train, test, target=TARGET)
print(dataset_profile.report())

In [ ]:
# Kolon bazli kompakt tablo -- hizli gozden gecirme icin
quick_look(train)

## 3 · Hedef dağılımı

Hedefin şekli üç kararı belirler: metrik, dönüşüm, model ailesi. Gerçek GDZ
verisinde hedefi `kesinti_dk = endtime − starttime` olarak kurduk; medyan 104 dk
ama maksimum 17.359 dk = 12.1 gün (`scripts/real_data_rehearsal.py`). Ağır sağ
kuyruk + sıfır yığılması birlikte görülür ve MAE-ailesi kayıpları ile iki aşamalı
(hurdle) adayları öne çıkarır — hangisinin kazandığı 02 numaralı notebook'ta
**ölçülmüş** olarak duruyor.

- Çarpıklık > 2 → `log1p` dönüşümünü dene
- Sıfır yığılması büyükse → iki aşamalı model adayı; kararı sezgi değil ölçüm versin

In [ ]:
target_values = train[TARGET].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(target_values, bins=60, color="#4C6EF5", edgecolor="white", linewidth=0.4)
axes[0].set_title("Ham dağılım")
axes[0].set_xlabel(TARGET)

positive = target_values[target_values > 0]
axes[1].hist(np.log1p(positive), bins=60, color="#12B886", edgecolor="white", linewidth=0.4)
axes[1].set_title("log1p (yalnızca pozitifler)")

axes[2].boxplot(target_values, vert=True, widths=0.5)
axes[2].set_title("Kutu grafiği — aykırı değerler")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

print(f"çarpıklık = {target_values.skew():.3f}")
print(f"sıfır oranı = {(target_values == 0).mean():.3%}")
print(target_values.describe())

## 4 · Eksik veri haritası

Eksikliğin **rastgele olup olmadığı** önemlidir. Bir kolon yalnızca belirli bir
dönemde veya belirli bir ilçede eksikse bu bir sinyaldir — doldurmadan önce
`_eksikti` bayrağı ekle. Doldurma bayrağının kendisi ise asla feature olmaz
(aşağıda, panel bölümünde neden).

In [ ]:
missing = (train.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing):
    fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * len(missing))))
    ax.barh(missing.index[::-1], missing.values[::-1], color="#FA5252")
    ax.set_xlabel("eksik %")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Eksik değer yok.")

## 5 · Türkçe tuzaklar — sessiz satır kaybı

İki tuzak da provada **gerçekten başımıza geldi**; ikisi de hata fırlatmaz,
sadece satır kaybettirir:

- `'İ'.lower()` iki kod noktası üretir (U+0069 U+0307), dolayısıyla
  `'İ'.lower() != 'i'`. İl/ilçe adıyla yapılan join sessizce 0 satır döner.
- Gerçek veride ilçe adı **nitelenmiş** geldi: `Köprübaşı / Manisa` (Köprübaşı
  hem Manisa'da hem Trabzon'da var). Referans tablomuz yalın `Köprübaşı` tutuyor;
  47 ilçenin 46'sı normalize eşleşti, bu tek ilçenin **284 kaydı** sessizce
  düşecekti. `strip_qualifier` + `join_key` sonrası eşleşme 68.257/68.257 = %100.0
  (`scripts/real_data_rehearsal.py`).

Ders: her join'den sonra satır sayısı ve eşleşme oranı **doğrulanır**; harici veri
(hava, nüfus) eklemeden önce bu hücre koşulur.

In [ ]:
text_columns = categorical_columns(train)

for column in text_columns[:10]:
    sample = train[column].dropna().astype(str).head(300)
    if any(has_combining_dot(v) for v in sample):
        print(f"! {column}: BİRLEŞİK NOKTA var — yanlış .lower() kullanılmış")
    else:
        print(f"  {column}: temiz ({train[column].nunique()} benzersiz)")

# Kanit 1: naif yaklasim basarisiz, join_key basarili
print("\n'İ'.lower() =", codepoints("İ".lower()), "->", "İ".lower() == "i")
print("join_key('İZMİR') == join_key('Izmir') ->", join_key("İZMİR") == join_key("Izmir"))

# Kanit 2: niteleyici eki ayri bir adimdir -- join_key kesmez, strip_qualifier keser
print("strip_qualifier('Köprübaşı / Manisa') ->", strip_qualifier("Köprübaşı / Manisa"))

## 6 · Panel kararı — olay kaydından ilçe × gün ızgarasına

Kesinti verisi **olay kaydıdır**: her satır bir arıza, saat damgalı (`14:23` gibi),
kesintisiz gün için satır yok. Model ise düzenli bir ilçe × gün paneli ister.
Bu dönüşümün iki ölçülmüş tuzağı var:

1. **Saat damgası ızgaraya oturtulmazsa hedef sessizce buharlaşır.** Günlük ızgara
   gece yarılarından oluşur; `14:23` damgalı kayıt merge'de hiçbir güne eşleşmez.
   Kontrollü ölçümde hedef kütlesinin **%90.4'ü** yok oldu; günde ~3 olaylı veride
   kayıp %99.8 (`src/gridup/panel.py`, `tests/test_panel_uydurma.py`). Hata yok,
   uyarı yok — model hep sıfır öğrenir. `build_panel` damgayı ızgaraya oturtur ve
   hedef kütlesini doğrular.
2. **Dolgu değeri hedefin türüne bağlıdır.** "Kayıt yok", sayım/süre hedefinde
   gerçek 0'dır; ölçüm hedefinde (tüketim, gerilim) bilinmeyendir → NaN. Gerçek GDZ
   panelinde 7.710 satır (%34.8) sıfır dolgu aldı ve `_dolduruldu` bayrağı **asla
   feature olmaz** — modelin "bu satır dolgu" bilgisini öğrenmesi sızıntıdır.

Gerçek veride sonuç: 68.257 kayıt → 47 ilçe × 472 gün = 22.184 satır, doluluk
%65.2, hedef kütlesi %100.00 korundu (`scripts/real_data_rehearsal.py`).

In [ ]:
# Olay kaydi -> panel donusumu. Yarisma verisi HAZIR panelse bu adim atlanir.
ENTITY_COLUMN = "ILCE_KOLONU"      # TODO: varlik anahtari (join_key'den gecmis olmali)
RAW_TIME_COLUMN = "TARIH_KOLONU"   # TODO: ham zaman damgasi kolonu

# Once OLC: doluluk %100'u asiyorsa ayni gunde birden cok olay var demektir.
# (Eski olcum bunu maskeliyordu ve %304.8 doluluk raporlamisti -- duzeltildi.)
kapsam = panel_coverage(train, entity_columns=[ENTITY_COLUMN], time_column=RAW_TIME_COLUMN)
print(f"beklenen {kapsam['expected_rows']:,.0f}  gercek {kapsam['actual_rows']:,.0f}"
      f"  doluluk %{kapsam['coverage'] * 100:.1f}")

panel = build_panel(
    train, entity_columns=[ENTITY_COLUMN], time_column=RAW_TIME_COLUMN,
    value_columns=[TARGET],
)
# Hedef kutlesi korunmali -- korunmadiysa izgaraya oturtma bozuk demektir.
# Gercek GDZ verisinde %100.00 olculdu; sapma varsa devam etmeden DUR.
korunan = panel[TARGET].sum() / train[TARGET].sum()
print(f"panel {panel.shape}   hedef kutlesi %{korunan * 100:.2f} (100 olmali)")

## 7 · Doğrulama şeması — yarışmanın kazanıldığı karar

Yanlış şema iki yönden öldürür: **iyimser CV** (sızıntı → leaderboard'da çöküş)
veya **gürültülü CV** (hangi değişikliğin işe yaradığı görünmez → public LB'ye
göre karar → shakeup). Zaman + panel verisinde seçimimiz
`purged_time_series_split` ve gerekçeleri ölçülü:

- **`test_span` = tahmin ufku.** 2023 GDZ Datathon birincisi
  `TimeSeriesSplit(n_splits=3, test_size=744)` kullandı; 744 saat = 31 gün =
  test bloğunun tam boyu (`src/gridup/validation.py`). CV, tahmin edilecek ufku
  birebir taklit etmelidir. Panelde satır sayısına göre eşit bölme, zaman
  uzunlukları eşit olmayan fold'lar üretir ve skorlar karşılaştırılamaz olur.
- **`embargo` bilinçli seçilir ve ufuktan küçük olmaz.** Kayan pencereli feature'lar
  fold sınırını aşarsa son train satırları ilk valid satırlarıyla aynı ham veriyi
  görür — sessiz bir iyimserlik. Kütüphane bu yüzden embargo'yu zorunlu parametre
  yapar; sessiz küçük varsayılan (~2 gün) tam da önlemeye çalıştığı sızıntıya izin
  veriyordu.
- Provadaki kurulum: embargo 31 gün, 4 fold × 31 gün; her fold'un valid'i
  47 ilçe × 31 gün = 1.457 satır (`scripts/real_data_rehearsal.py`).

In [ ]:
TIME_COLUMN = None   # TODO: zaman kolonu (panel kurduysan panelin gun kolonu)

suggestion = suggest_scheme(train, target=TARGET)
print(suggestion)

if TIME_COLUMN:
    train_times = pd.to_datetime(train[TIME_COLUMN])
    test_times = pd.to_datetime(test[TIME_COLUMN])
    print(f"train: {train_times.min()} -> {train_times.max()}")
    print(f"test:  {test_times.min()} -> {test_times.max()}")
    print(f"bosluk: {test_times.min() - train_times.max()}")

    # UFUK = train'in son gunu -> test blogunun son gunu (bosluk DAHIL).
    # (test.max - test.min + 1) formulu boslugu yok sayar: 10 gunluk boslukta
    # CV lag'i 20 gun, test lag'i 30 gun bayatti (olculdu 2026-08-18).
    # AMBARGO = train-test boslugu; bitisikse 0. Gecmis-hedef feature'lari
    # zaten horizon kadar kaydirildigi icin fazladan ambargo CV'yi dagitim
    # rejiminden UZAKLASTIRIR (her fold 30+ gun bayat egitim gorur).
    geo = forecast_geometry(train_times, test_times)
    HORIZON, EMBARGO_DAYS = geo.horizon_days, geo.gap_days
    print(geo.summary())
    folds = purged_time_series_split(
        train_times, n_splits=4,
        embargo=pd.Timedelta(days=EMBARGO_DAYS),
        test_span=pd.Timedelta(days=HORIZON),
    )
    for i, (tr, va) in enumerate(folds, 1):
        print(f"fold {i}: train={len(tr):>8,}  valid={len(va):>6,}")

## 8 · Sızıntı duvarı

Kesinti verisinde en tehlikeli kolonlar **aynı günün bilgisini** taşıyanlardır:
arıza sebebi, etkilenen abone sayısı, o günkü yük. Tahmin anında bunlar bilinmez.
Provada bunu bilerek ölçtük: aynı-gün kolonları feature bırakıldığında gain
tablosunun tepesine oturuyorlar — `effectedsubscribers` tek başına en iyi meşru
feature'ın (`sicaklik_max`) yaklaşık **24 katı** gain topladı; ID kolonu bile ~8
katıyla ikinci sıradaydı (`scripts/real_data_rehearsal.py`). Böyle bir model CV'de
parlar, gerçek tahmin gününde o kolonlar olmadığı için çöker.

Duvarın üç kuralı:

1. Aynı günün bilgisi feature olamaz; yalnızca ufuk kadar kaydırılmış geçmiş
   agregatları (lag/rolling) meşrudur.
2. Hedeften türetilen hiçbir şey aynı satırın feature'ı olamaz.
3. `leakage_report` model eğitilmeden **önce** koşulur; `critical` bulgu varsa
   çözülmeden devam edilmez.

In [ ]:
findings = leakage_report(train, TARGET, test=test, time_column=TIME_COLUMN)
print(findings["summary"], "\n")

for severity in ("critical", "warning", "info"):
    for message in findings[severity]:
        print(f"[{severity.upper()}] {message}")

## Çıktı: üç karar

> Bu tablo veri gününde doldurulur — jüri "veriyi anladık" iddiasının kanıtını
> burada görür. Prova satırı, doldurulmuş bir örnek olarak bırakıldı.

| Karar | Prova (gerçek GDZ, ölçüldü) | Yarışma verisi (doldur) |
|---|---|---|
| Hedef + panel | `kesinti_dk`; 47 ilçe × 472 gün; kütle %100.00 | … |
| CV şeması | purged, embargo 31 g, 4 fold × 31 g `test_span` | … |
| Sızıntı duvarı | sebep / abone / yük → yalnız ufuk-kaydırmalı lag | … |

## Kaynaklar ve Atıf

Bu çalışmada kullanılan dış veri kaynakları ve lisansları:

| Kaynak | Lisans | Atıf |
|---|---|---|
| Open-Meteo (hava durumu, arşiv + tahmin) | CC BY 4.0 | Weather data by Open-Meteo.com |
| ESA WorldCover 10m v200 (arazi örtüsü) | CC BY 4.0 | © ESA WorldCover project 2021 |
| OpenStreetMap (`power=*` altyapı) | ODbL 1.0 | © OpenStreetMap contributors |
| TÜİK (turizm istatistikleri) | Kamuya açık | Türkiye İstatistik Kurumu |
| AFAD (deprem kataloğu) | Kamuya açık | AFAD Deprem Dairesi Başkanlığı |
| NASA FIRMS (yangın tespitleri) | Kamuya açık | NASA FIRMS / MODIS-VIIRS |

Veri kökeni, SHA-256 özetleri ve yeniden dağıtım kararları `data/sources.yml`
manifestinde tutulur; `scripts/veri_sagligi.py` her kaynağın kapsam, bütünlük
ve fizik kontrollerini koşar.

**Yarışma hedefinin geçmişi (EPİAŞ plansız kesinti kayıtları) modele girdi
olarak KULLANILMAMIŞTIR.** Bu veri yalnızca boru hattının gerçek veri
üzerinde prova edilmesinde kullanılmış, model girdisi olmasını engelleyen
kapı `src/gridup/uygunluk.py` içinde kod düzeyinde zorlanmaktadır
(`tests/test_uygunluk_kapisi.py`).